# PM Assistant Demo

Minimal end-to-end demo for `llm_router_agent`.

Prereq (from `llm-router-agent/`):

```bash
python -m venv .venv
source .venv/bin/activate
pip install -e .
```

Note: `scoper` is always routed to Groq. Set `GROQ_API_KEY` even when `LLM_PROVIDER=ollama`.


In [1]:
import os

from dotenv import load_dotenv

from llm_router_agent import Team, TeamMember, build_graph


In [2]:
load_dotenv()

provider = os.getenv("LLM_PROVIDER", "ollama")
ollama_model = os.getenv("OLLAMA_MODEL", "llama3.1")
groq_model = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
groq_key = os.getenv("GROQ_API_KEY", "").strip()

print(f"LLM_PROVIDER={provider}")
print(f"OLLAMA_MODEL={ollama_model}")
print(f"GROQ_MODEL={groq_model}")

if not groq_key:
    raise EnvironmentError(
        "GROQ_API_KEY is required because scoper runs on Groq by policy."
    )


LLM_PROVIDER=ollama
OLLAMA_MODEL=llama3.1
GROQ_MODEL=llama-3.3-70b-versatile


In [3]:
RUN_DEEP_REVIEW = False

project_description = "Create a secure, multi-user AI dashboard using LangGraph and React."

team = Team(
    team_members=[
        TeamMember(name="Alice", role="Lead Dev", skills=["Python", "LangGraph", "Architecture"], seniority="Senior"),
        TeamMember(name="Bob", role="Frontend", skills=["React", "UI/UX"], seniority="Mid"),
        TeamMember(name="Charlie", role="QA", skills=["Testing", "Python"], seniority="Junior"),
    ]
)

initial_state = {
    "project_description": project_description,
    "team": team,
    "iteration_number": 0,
    "max_iteration": 2,
    "insights": [],
    "project_risk_score_iterations": [],
}


In [ ]:
graph = build_graph()
final_state = graph.invoke(initial_state, {"configurable": {"thread_id": "pm-assistant-demo"}})


In [ ]:
print("Tasks:")
for task in final_state["tasks"].task:
    print(f"- {task.id}: {task.task_name} ({task.estimated_day}d, skill={task.required_skill})")

print("\nDependencies:")
for dep in final_state.get("dependencies", []):
    print(f"- {dep.task_id} depends on {dep.dependent_on}")

print("\nSchedule:")
schedule = final_state.get("schedule")
if schedule:
    for item in schedule.schedule:
        print(f"- {item.task.task_name}: day {item.start_day} -> {item.end_day}")

print("\nAllocations:")
allocations = final_state.get("task_allocations")
if allocations:
    for alloc in allocations.task_allocations:
        print(f"- {alloc.task.task_name}: {alloc.team_member.name}")

print("\nRisks:")
risks = final_state.get("risks")
if risks:
    for risk in risks.risks:
        print(f"- {risk.task_name}: {risk.score} ({risk.reason})")

print("\nInsights:")
for insight in final_state.get("insights", []):
    print(f"- {insight}")


In [ ]:
# Optional: Plotly Gantt (only if plotly + pandas are installed)
try:
    import pandas as pd
    import plotly.express as px

    schedule = final_state.get("schedule")
    allocations = final_state.get("task_allocations")

    if schedule and allocations:
        rows = []
        alloc_by_task = {a.task.task_name: a.team_member.name for a in allocations.task_allocations}
        for s in schedule.schedule:
            rows.append({
                "Task": s.task.task_name,
                "Start": s.start_day,
                "Finish": s.end_day,
                "Assignee": alloc_by_task.get(s.task.task_name, "Unassigned"),
            })

        if rows:
            df = pd.DataFrame(rows)
            fig = px.timeline(df, x_start="Start", x_end="Finish", y="Task", color="Assignee", title="Project Schedule")
            fig.update_yaxes(autorange="reversed")
            fig.show()
        else:
            print("No schedule rows to plot.")
    else:
        print("Missing schedule or allocations; skipping chart.")
except Exception:
    print("Plotly/pandas not installed; skipping Gantt chart.")


In [ ]:
# Optional self-review toggle
if RUN_DEEP_REVIEW:
    print("Deep review enabled. Add your custom review prompt/model call here.")
